# llm_local · 02 — Fine-tuning del modello piccolo

Insegna a **Qwen 2.5 da 0,5 miliardi di parametri** a scrivere la sintesi finale della lettura,
studiando le sintesi scritte da Gemma (`sintesi_pulite.jsonl`).

Alla fine, nella sezione **Output**, trovi:
- `tarocchi-qwen0.5b-q8_0.gguf` — il modello pronto per Ollama (circa 500 MB, gira anche senza GPU)
- `Modelfile` — le istruzioni per caricarlo in Ollama
- `adattatore_lora/` — solo i pesi imparati (piccolo, utile per riaddestrare)
- `prove.txt` — le sintesi di prova, prima e dopo l'addestramento

### Prima di avviare (pannello a destra)
- **Accelerator**: `GPU T4 x2` (ne usiamo una) oppure `GPU P100`
- **Internet**: `On`
- **Input**: il dataset con `sintesi_pulite.jsonl`

In [ ]:
# ===== IMPOSTAZIONI =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # una sola GPU: più semplice e stabile
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # meno frammentazione della memoria

MODELLO_BASE = "Qwen/Qwen2.5-0.5B-Instruct"
NOME         = "tarocchi-qwen0.5b"
EPOCHE       = 3
LR           = 2e-4
LORA_R       = 16
BATCH        = 1       # esempi elaborati insieme (1 = minimo consumo di memoria)
ACCUMULO     = 16      # esempi accumulati prima di ogni aggiornamento dei pesi
MAX_TOKEN    = 1536
N_VALIDAZIONE = 40      # sintesi tenute da parte per controllare l'apprendimento
OUT          = "/kaggle/working"

from datetime import datetime
from zoneinfo import ZoneInfo
def fatto(cella):
    print(f"\n✅ Completata: {cella} — ore {datetime.now(ZoneInfo('Europe/Rome')):%H:%M:%S}")

fatto("Impostazioni")

## 1. Librerie

In [ ]:
# torchao preinstallato su Kaggle è troppo vecchio e peft lo rifiuta: non ci serve, lo tolgo
!pip uninstall -y -q torchao
!pip install -q peft
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0))

fatto("Installazione librerie")

## 2. Dataset

In [ ]:
import glob, json, random

trovati = glob.glob("/kaggle/input/**/sintesi_pulite.jsonl", recursive=True)
assert trovati, "Non trovo sintesi_pulite.jsonl: aggiungilo nel pannello Input"
with open(trovati[0], encoding="utf-8") as f:
    righe = [json.loads(r) for r in f if r.strip()]

random.Random(0).shuffle(righe)
valid, train = righe[:N_VALIDAZIONE], righe[N_VALIDAZIONE:]
SYSTEM = righe[0]["messages"][0]["content"]
print(f"Esempi: {len(righe)}  (addestramento {len(train)}, validazione {len(valid)})")
print("Istruzione di sistema:", SYSTEM)

fatto("Caricamento dataset")

## 3. Modello base

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODELLO_BASE)
# pesi in float32: con LoRA e fp16 misto è la combinazione più stabile sulle T4
modello = AutoModelForCausalLM.from_pretrained(MODELLO_BASE, dtype=torch.float32).to("cuda")
print(f"Parametri: {modello.num_parameters()/1e6:.0f} milioni")

fatto("Caricamento modello base")

## 4. Prova PRIMA dell'addestramento
Così vediamo da dove partiamo.

In [ ]:
def sintesi(modello, user, temperatura=0.8):
    msg = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]
    # return_dict=True: funziona sia con le versioni vecchie sia con le nuove di transformers
    ingressi = tok.apply_chat_template(msg, add_generation_prompt=True, return_tensors="pt",
                                       return_dict=True).to(modello.device)
    with torch.no_grad():
        out = modello.generate(**ingressi, max_new_tokens=350, do_sample=True, temperature=temperatura,
                               top_p=0.9, repetition_penalty=1.1, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ingressi["input_ids"].shape[1]:], skip_special_tokens=True).strip()

PROVE = valid[:3]
registro = open(f"{OUT}/prove.txt", "w", encoding="utf-8")
modello.eval()
for es in PROVE:
    testo = sintesi(modello, es["messages"][1]["content"])
    blocco = f"=== PRIMA · {es['chiave']} ===\n{testo}\n"
    print(blocco); registro.write(blocco + "\n")

fatto("Prova del modello PRIMA dell'addestramento")

## 5. Preparazione degli esempi
Il modello impara solo dalla parte della risposta (la sintesi): le carte in ingresso non contano nella perdita.

In [ ]:
from datasets import Dataset

def prepara(es):
    msg = es["messages"]
    prompt = tok.apply_chat_template(msg[:2], tokenize=False, add_generation_prompt=True)
    completo = tok.apply_chat_template(msg, tokenize=False)
    assert completo.startswith(prompt)
    ids_prompt = tok(prompt, add_special_tokens=False)["input_ids"]
    ids = tok(completo, add_special_tokens=False)["input_ids"][:MAX_TOKEN]
    etichette = [-100] * len(ids_prompt) + ids[len(ids_prompt):]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": etichette[:len(ids)]}

ds_train = Dataset.from_list(train).map(prepara, remove_columns=list(train[0].keys()))
ds_valid = Dataset.from_list(valid).map(prepara, remove_columns=list(valid[0].keys()))
lung = [len(x) for x in ds_train["input_ids"]]
print(f"Token per esempio: min {min(lung)}, medio {sum(lung)//len(lung)}, max {max(lung)}")

fatto("Preparazione esempi")

## 6. Addestramento (LoRA)
Circa 15-30 minuti. Ogni 10 passi viene stampata la *loss*: deve scendere.

In [ ]:
from peft import LoraConfig, get_peft_model
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

lora = LoraConfig(r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
modello.gradient_checkpointing_enable()
modello.enable_input_require_grads()      # necessario per il checkpointing con LoRA
modello = get_peft_model(modello, lora)
modello.print_trainable_parameters()

import math
passi_totali = math.ceil(len(ds_train) / (BATCH * ACCUMULO)) * EPOCHE
passi_riscaldamento = max(1, int(0.05 * passi_totali))           # 5% dei passi
print(f"Passi di addestramento: {passi_totali} (riscaldamento {passi_riscaldamento})")

args = TrainingArguments(
    output_dir=f"{OUT}/checkpoint", num_train_epochs=EPOCHE, learning_rate=LR,
    per_device_train_batch_size=BATCH, per_device_eval_batch_size=BATCH, gradient_accumulation_steps=ACCUMULO,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine", warmup_steps=passi_riscaldamento, weight_decay=0.0,
    fp16=True, logging_steps=10, eval_strategy="epoch", save_strategy="no",
    report_to="none", disable_tqdm=True, seed=42,
)
trainer = Trainer(model=modello, args=args, train_dataset=ds_train, eval_dataset=ds_valid,
                  data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100))
t0 = datetime.now()
trainer.train()
print(f"Durata addestramento: {(datetime.now() - t0).seconds // 60} minuti")
for r in trainer.state.log_history:
    if "eval_loss" in r:
        print(f"Epoca {r['epoch']:.0f}: loss di validazione {r['eval_loss']:.3f}")

fatto("Addestramento")

## 7. Prova DOPO l'addestramento
Stesse carte di prima. L'ultima estrazione viene ripetuta due volte: le due sintesi devono essere **diverse** ma con lo stesso significato.

In [ ]:
modello.eval()
modello.config.use_cache = True
for es in PROVE + [PROVE[-1]]:
    testo = sintesi(modello, es["messages"][1]["content"])
    blocco = f"=== DOPO · {es['chiave']} · {len(testo.split())} parole ===\n{testo}\n"
    print(blocco); registro.write(blocco + "\n")
riferimento = f"=== GEMMA (riferimento) · {PROVE[0]['chiave']} ===\n{PROVE[0]['messages'][2]['content']}\n"
print(riferimento); registro.write(riferimento); registro.close()

fatto("Prova del modello DOPO l'addestramento")

## 8. Unione e salvataggio
L'adattatore LoRA viene fuso nel modello base, ottenendo un modello unico.

In [ ]:
modello.save_pretrained(f"{OUT}/adattatore_lora")
unito = modello.merge_and_unload().to(torch.float16)
unito.save_pretrained(f"{OUT}/unito", safe_serialization=True)
tok.save_pretrained(f"{OUT}/unito")
!du -sh {OUT}/unito {OUT}/adattatore_lora

fatto("Unione e salvataggio")

## 9. Conversione in GGUF (formato di Ollama)
`q8_0`: quasi nessuna perdita di qualità e circa metà della dimensione.

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/temp/llama.cpp
!pip install -q gguf sentencepiece protobuf
GGUF = f"{OUT}/{NOME}-q8_0.gguf"
!python /kaggle/temp/llama.cpp/convert_hf_to_gguf.py {OUT}/unito --outtype q8_0 --outfile {GGUF}
!ls -lh {GGUF}

modelfile = f'''FROM ./{NOME}-q8_0.gguf

TEMPLATE """{{{{ if .System }}}}<|im_start|>system
{{{{ .System }}}}<|im_end|>
{{{{ end }}}}{{{{ if .Prompt }}}}<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
{{{{ end }}}}<|im_start|>assistant
"""

SYSTEM """{SYSTEM}"""

PARAMETER temperature 0.8
PARAMETER top_p 0.9
PARAMETER repeat_penalty 1.1
PARAMETER num_predict 350
PARAMETER num_ctx 2048
PARAMETER stop "<|im_end|>"
PARAMETER stop "<|endoftext|>"
'''
with open(f"{OUT}/Modelfile", "w", encoding="utf-8") as f:
    f.write(modelfile)
print(modelfile)

fatto("Conversione GGUF")

## 10. Prova finale con Ollama
Carichiamo il GGUF in Ollama, esattamente come farai sul server, e generiamo due sintesi con le stesse carte.

In [ ]:
import subprocess, time, requests
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)
!cd {OUT} && ollama create {NOME} -f Modelfile

utente = PROVE[1]["messages"][1]["content"]
for i in range(2):
    t0 = time.time()
    r = requests.post("http://127.0.0.1:11434/api/generate", timeout=300,
                      json={"model": NOME, "prompt": utente, "stream": False}).json()
    print(f"--- Ollama, tentativo {i+1} ({time.time()-t0:.1f} s) ---\n{r['response'].strip()}\n")
server.terminate()

fatto("Prova finale con Ollama")

## 11. Pulizia
Tolgo i file intermedi pesanti, così nell'Output restano solo quelli utili.

In [ ]:
!rm -rf {OUT}/unito {OUT}/checkpoint
!ls -lh {OUT}

fatto("Pulizia output")